### Earthquake PINN Training on Colab GPU
This notebook clones the repository, installs dependencies, and runs the full-scale training.
It also includes optional Hyperparameter Optimization (Optuna).

**Note**: Make sure you have pushed your local changes (`git push origin dev`) before running this.

In [48]:
!nvidia-smi

Mon Feb 16 01:45:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [49]:
# Install uv and clean up existing clone
!pip install uv
import os
%cd /content
!rm -rf earthquake_proj
!git clone -b dev --single-branch https://github.com/sattary/earthquake_proj.git
%cd earthquake_proj

Cloning into 'earthquake_proj'...
remote: Enumerating objects: 286, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 286 (delta 94), reused 261 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (286/286), 20.49 MiB | 16.94 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Filtering content: 100% (30/30), 128.06 MiB | 31.97 MiB/s, done.
/content/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj


In [50]:
# Sync environment
!uv sync

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 104 packages in 0.91ms
Prepared 1 package in 28ms                                               
Installed 98 packages in 425ms                              
 + alembic==1.18.4
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + certifi==2026.1.4
 + click==8.3.1
 + cloudpickle==3.1.2
 + colorlog==6.10.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.4
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + executing==2.2.1
 + filelock==3.24.0
 + fonttools==4.61.1
 + fsspec==2026.2.0
 + greenlet==3.3.1
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.10.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.19.2
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.4.9
 + llvmlite==0.46.0
 + mako==1.3.10
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + matplotlib-inline==0.2.1
 + mdurl==

In [51]:
!ls

checkpoints  docs     notebooks       README.md  src	uv.lock
data	     main.py  pyproject.toml  results	 tests


In [52]:
import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figs", exist_ok=True)

# Ensure package initialization for -m calls
if os.path.exists('src'):
    if not os.path.exists('src/__init__.py'):
        open('src/__init__.py', 'a').close()
        print("Initialized 'src' as package.")
elif os.path.exists('earthquake_proj'):
    if not os.path.exists('earthquake_proj/__init__.py'):
        open('earthquake_proj/__init__.py', 'a').close()
        print("Initialized 'earthquake_proj' as package.")

Initialized 'src' as package.


In [73]:
# Step 1: Tune Hyperparameters (Optional - skip if using defaults)
!PYTHONPATH=. uv run python -m src.pipelines.cli tune --n-trials 50 --epochs 500


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/src/pipelines/cli.py", line 10, in <module>
    from src.pipelines.inference import plot as run_plot
ImportError: cannot import name 'plot' from 'src.pipelines.inference' (/content/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/src/pipelines/inference.py). Did you mean: 'plt'?


In [ ]:
# Step 2: Final Stabilized Training
!PYTHONPATH=. uv run python -m src.pipelines.cli train --epochs 20000 --n-coll 20000 --config results/tables/best_params.json


In [ ]:
# Step 3: Generate Academic Figure Suite (PDF, SVG, High-Res PNG)
!PYTHONPATH=. uv run python -m src.pipelines.cli results-suite


In [ ]:
from IPython.display import Image, display
import os
for f in ['results/figs/stress_map_10km.png', 'results/figs/velocity_map_10km.png']:
    if os.path.exists(f): display(Image(f))

In [56]:
# Pack Everything for Download/Active Storage
!zip -r results_archive.zip results/ checkpoints/final_model.pth
try:
  from google.colab import files
  files.download('results_archive.zip')
except ImportError:
  print('Not running on Google Colab, skipping download.')

updating: results/ (stored 0%)
updating: results/figs/ (stored 0%)
updating: results/figs/eda/ (stored 0%)
updating: results/figs/eda/azimuth_rose.png (deflated 9%)
updating: results/figs/eda/eda_l3_test/ (stored 0%)
updating: results/figs/eda/eda_l3_test/azimuth_rose.png (deflated 9%)
updating: results/figs/eda/eda_l3_test/gps_density.png (deflated 14%)
updating: results/figs/eda/eda_l3_test/velocity_slices.png (deflated 23%)
updating: results/figs/eda/gps_density.png (deflated 14%)
updating: results/figs/eda/velocity_slices.png (deflated 23%)
updating: results/tables/ (stored 0%)
updating: checkpoints/final_model.pth (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [67]:
!ls


checkpoints  docs     notebooks       README.md  results_archive.zip  tests
data	     main.py  pyproject.toml  results	 src		      uv.lock


In [68]:
cp -R results_archive.zip /content/drive/MyDrive/

In [70]:
!ls /content/drive/MyDrive/ | grep .zip

results_archive.zip
